In [5]:
# Author: Robyn Martin
# Project: CHF Mortality Risk Analysis
# Date: April 2026

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [2]:
#instalation of the pandas library helps with reading files
import pandas as pd
from google.colab import files
uploaded = files.upload()

Saving heart_failure_clinical_records_dataset.csv to heart_failure_clinical_records_dataset.csv


In [3]:
df = pd.read_csv("heart_failure_clinical_records_dataset.csv")

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Prepare CHF data
# -----------------------------
df = df.copy()

if "age_group" not in df.columns:
    df["age_group"] = pd.cut(
        df["age"],
        bins=[0, 49, 59, 69, 200],
        labels=["<50", "50-59", "60-69", "70+"],
        include_lowest=True
    )

if "low_EF" not in df.columns:
    df["low_EF"] = (df["ejection_fraction"] < 35).astype(int)

if "high_creatinine" not in df.columns:
    df["high_creatinine"] = (df["serum_creatinine"] > 1.5).astype(int)

if "low_sodium" not in df.columns:
    df["low_sodium"] = (df["serum_sodium"] < 135).astype(int)

baseline_risk = df["DEATH_EVENT"].mean()
baseline_n = len(df)
baseline_deaths = int(df["DEATH_EVENT"].sum())

# -----------------------------
# Widgets
# -----------------------------
age_widget = widgets.Dropdown(
    options=["Any", "<50", "50-59", "60-69", "70+"],
    value="Any",
    description="Age"
)

ef_widget = widgets.Dropdown(
    options=["Any", "Yes", "No"],
    value="Any",
    description="Low EF"
)

na_widget = widgets.Dropdown(
    options=["Any", "Yes", "No"],
    value="Any",
    description="Low Na"
)

anaemia_widget = widgets.Dropdown(
    options=["Any", "Yes", "No"],
    value="Any",
    description="Anaemia"
)

creatinine_widget = widgets.Dropdown(
    options=["Any", "Yes", "No"],
    value="Any",
    description="High Cr"
)

smoking_widget = widgets.Dropdown(
    options=["Any", "Yes", "No"],
    value="Any",
    description="Smoking"
)

output = widgets.Output()

# -----------------------------
# Selection order tracking
# -----------------------------
selection_order = []

def update_selection_order(label, widget):
    global selection_order

    # remove existing entry for this label
    selection_order = [item for item in selection_order if item[0] != label]

    # add back only if selected
    if widget.value != "Any":
        selection_order.append((label, widget.value))

def get_selected_filters_in_order():
    return selection_order.copy()

# -----------------------------
# Helper functions
# -----------------------------
def yes_no_to_binary(value):
    if value == "Yes":
        return 1
    if value == "No":
        return 0
    return None

def summarize_group(dataframe):
    n = len(dataframe)
    deaths = int(dataframe["DEATH_EVENT"].sum()) if n > 0 else 0
    risk = deaths / n if n > 0 else np.nan
    return n, deaths, risk

def apply_selected_filters(dataframe):
    filtered = dataframe.copy()

    if age_widget.value != "Any":
        filtered = filtered[filtered["age_group"].astype(str) == age_widget.value]

    if ef_widget.value != "Any":
        filtered = filtered[filtered["low_EF"] == yes_no_to_binary(ef_widget.value)]

    if na_widget.value != "Any":
        filtered = filtered[filtered["low_sodium"] == yes_no_to_binary(na_widget.value)]

    if anaemia_widget.value != "Any":
        filtered = filtered[filtered["anaemia"] == yes_no_to_binary(anaemia_widget.value)]

    if creatinine_widget.value != "Any":
        filtered = filtered[filtered["high_creatinine"] == yes_no_to_binary(creatinine_widget.value)]

    if smoking_widget.value != "Any":
        filtered = filtered[filtered["smoking"] == yes_no_to_binary(smoking_widget.value)]

    return filtered

def apply_one_named_filter(dataframe, label, value):
    filtered = dataframe.copy()

    if label == "Age":
        filtered = filtered[filtered["age_group"].astype(str) == value]
    elif label == "Low EF":
        filtered = filtered[filtered["low_EF"] == yes_no_to_binary(value)]
    elif label == "Low Na":
        filtered = filtered[filtered["low_sodium"] == yes_no_to_binary(value)]
    elif label == "Anaemia":
        filtered = filtered[filtered["anaemia"] == yes_no_to_binary(value)]
    elif label == "High Cr":
        filtered = filtered[filtered["high_creatinine"] == yes_no_to_binary(value)]
    elif label == "Smoking":
        filtered = filtered[filtered["smoking"] == yes_no_to_binary(value)]

    return filtered

# -----------------------------
# Chart functions
# -----------------------------
def make_main_comparison_chart(overall_risk, selected_risk):
    labels = ["Overall Cohort", "Selected Profile"]
    values = [
        overall_risk * 100,
        selected_risk * 100 if pd.notna(selected_risk) else 0
    ]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(labels, values)

    ax.set_title("Mortality Risk Comparison")
    ax.set_ylabel("Mortality Rate (%)")
    ax.set_ylim(0, max(values + [10]) * 1.25)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{val:.1f}%",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

def make_added_variable_chart(dataframe, selected_filters):
    if len(selected_filters) == 0:
        print("\nNo filters selected.")
        return

    if len(selected_filters) == 1:
        print("\nSelect at least 2 filters to show the added-variable comparison chart.")
        return

    # first selected filter becomes the base
    base_label, base_value = selected_filters[0]

    current_df = apply_one_named_filter(dataframe, base_label, base_value)
    base_n, base_d, base_risk = summarize_group(current_df)

    if base_n == 0 or pd.isna(base_risk):
        print("\nNo patients match the base factor.")
        return

    labels = [f"{base_label}"]
    base_vals = [base_risk * 100]
    added_vals = [0]
    total_vals = [base_risk * 100]
    annotations = [f"n={base_n}, d={base_d}"]

    previous_risk_pct = base_risk * 100
    cumulative_label = base_label

    # build cumulatively in true selection order
    for add_label, add_value in selected_filters[1:]:
        current_df = apply_one_named_filter(current_df, add_label, add_value)
        combo_n, combo_d, combo_risk = summarize_group(current_df)

        combo_pct = combo_risk * 100 if pd.notna(combo_risk) else 0
        added_pct = combo_pct - previous_risk_pct

        cumulative_label = f"{cumulative_label} + {add_label}"
        labels.append(cumulative_label)
        base_vals.append(previous_risk_pct)
        added_vals.append(added_pct)
        total_vals.append(combo_pct)
        annotations.append(f"n={combo_n}, d={combo_d}")

        previous_risk_pct = combo_pct

    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(11, 5))

    added_positive = [max(v, 0) for v in added_vals]
    added_negative = [min(v, 0) for v in added_vals]

    # draw bars
    ax.bar(x, base_vals, label="Previous mortality level", color="#1f77b4")
    ax.bar(x, added_positive, bottom=base_vals, label="Added mortality", color="#ff7f0e")
    ax.bar(x, added_negative, bottom=base_vals, color="#ff7f0e", label="_nolegend_")

    # total mortality labels on top
    for i, total in enumerate(total_vals):
        ax.text(
            i,
            total + 2,
            f"{total:.0f}%",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

    # added mortality labels inside orange segment
    for i, (base, added) in enumerate(zip(base_vals, added_vals)):
        if abs(added) > 0.5:
            ax.text(
                i,
                base + (added * 0.6),
                f"+{added:.0f}%" if added > 0 else f"{added:.0f}%",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if abs(added) >= 8 else "black",
                fontweight="bold"
            )

    # n and d labels above bars
    for i, (total, note) in enumerate(zip(total_vals, annotations)):
        ax.text(
            i,
            total + 9,
            note,
            ha="center",
            va="bottom",
            fontsize=8
        )

    ax.axhline(
        baseline_risk * 100,
        linestyle="--",
        linewidth=1,
        color="gray"
    )

    ax.set_title("Cumulative Added-Variable Mortality Chart")
    ax.set_ylabel("Mortality Rate (%)")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.legend()

    min_y = min(total_vals + [0])
    max_y = max(total_vals + [10])
    ax.set_ylim(min(min_y - 5, 0), max_y * 1.35)

    plt.tight_layout()
    plt.show()

# -----------------------------
# Main update function
# -----------------------------
def update_view(*args):
    with output:
        clear_output(wait=True)

        filtered = apply_selected_filters(df)
        n, deaths, risk = summarize_group(filtered)

        print("CHF What-If Risk Explorer")
        print("-" * 40)
        print(f"Matching patients: {n}")
        print(f"Deaths: {deaths}")

        if n > 0:
            print(f"Observed mortality risk: {risk:.1%}")
            print(f"Overall cohort mortality: {baseline_risk:.1%}")
            print(f"Difference from baseline: {(risk - baseline_risk):+.1%}")
        else:
            print("No patients match this profile.")

        if 0 < n < 10:
            print("\nCaution: very small subgroup.")
        elif 10 <= n < 20:
            print("\nNote: subgroup is fairly small; interpret with caution.")

        # Chart 1
        make_main_comparison_chart(baseline_risk, risk)

        # Chart 2
        selected_filters = get_selected_filters_in_order()
        make_added_variable_chart(df, selected_filters)

# -----------------------------
# Widget change handlers
# -----------------------------
def on_age_change(change):
    if change["name"] == "value":
        update_selection_order("Age", age_widget)
        update_view()

def on_ef_change(change):
    if change["name"] == "value":
        update_selection_order("Low EF", ef_widget)
        update_view()

def on_na_change(change):
    if change["name"] == "value":
        update_selection_order("Low Na", na_widget)
        update_view()

def on_anaemia_change(change):
    if change["name"] == "value":
        update_selection_order("Anaemia", anaemia_widget)
        update_view()

def on_creatinine_change(change):
    if change["name"] == "value":
        update_selection_order("High Cr", creatinine_widget)
        update_view()

def on_smoking_change(change):
    if change["name"] == "value":
        update_selection_order("Smoking", smoking_widget)
        update_view()

# -----------------------------
# Connect widget changes
# -----------------------------
age_widget.observe(on_age_change, names="value")
ef_widget.observe(on_ef_change, names="value")
na_widget.observe(on_na_change, names="value")
anaemia_widget.observe(on_anaemia_change, names="value")
creatinine_widget.observe(on_creatinine_change, names="value")
smoking_widget.observe(on_smoking_change, names="value")

controls = widgets.VBox([
    age_widget,
    ef_widget,
    na_widget,
    anaemia_widget,
    creatinine_widget,
    smoking_widget
])

display(widgets.HBox([controls]))
display(output)

update_view()

Output()